# View GSO-SLAM Results in Jupyter

This notebook loads `gs_map/point_cloud.ply` and `gs_map/input.ply` from completed runs under `experiments_bash/results/test`.

It tries Open3D first. If the notebook viewer is unavailable, it falls back to inline Plotly, then Matplotlib.


In [ ]:
from pathlib import Path

import numpy as np
import open3d as o3d
import plotly.graph_objects as go
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "experiments_bash").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd())
RESULT_ROOT = REPO_ROOT / "experiments_bash" / "results" / "test"
print(f"Repo root: {REPO_ROOT}")
print(f"Result root: {RESULT_ROOT}")


In [ ]:
def list_run_dirs(result_root: Path) -> list[Path]:
    run_dirs = set()
    for ply_path in result_root.glob("**/gs_map/point_cloud.ply"):
        run_dirs.add(ply_path.parent.parent)
    for ply_path in result_root.glob("**/gs_map/input.ply"):
        run_dirs.add(ply_path.parent.parent)
    return sorted(run_dirs)

run_dirs = list_run_dirs(RESULT_ROOT)
for idx, run_dir in enumerate(run_dirs):
    print(f"[{idx}] {run_dir.relative_to(REPO_ROOT)}")

RUN_NAME = "replica_room0_quality_codexfix"  # change this to the run you want
RUN_DIR = next((run_dir for run_dir in run_dirs if run_dir.name == RUN_NAME), None)
if RUN_DIR is None:
    raise FileNotFoundError(f"Could not find a run named {RUN_NAME!r} under {RESULT_ROOT}")

POINT_CLOUD_PLY = RUN_DIR / "gs_map" / "point_cloud.ply"
GS_SPLAT_PLY = RUN_DIR / "gs_map" / "input.ply"
print(f"Selected run: {RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Point cloud: {POINT_CLOUD_PLY}")
print(f"GS splat:   {GS_SPLAT_PLY}")


In [ ]:
def load_point_cloud(path: Path, max_points: int | None = 500_000) -> o3d.geometry.PointCloud:
    pcd = o3d.io.read_point_cloud(str(path))
    if pcd.is_empty():
        raise RuntimeError(f"Open3D could not read any points from {path}")

    if max_points is not None and len(pcd.points) > max_points:
        ratio = max_points / float(len(pcd.points))
        pcd = pcd.random_down_sample(ratio)

    return pcd

def pcd_to_arrays(pcd: o3d.geometry.PointCloud) -> tuple[np.ndarray, np.ndarray | None]:
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else None
    return points, colors

def show_plotly(points: np.ndarray, colors: np.ndarray | None, title: str) -> None:
    sample = points
    sample_colors = colors
    max_points = 100_000
    if len(sample) > max_points:
        idx = np.random.choice(len(sample), max_points, replace=False)
        sample = sample[idx]
        sample_colors = sample_colors[idx] if sample_colors is not None else None

    if sample_colors is not None and len(sample_colors) == len(sample):
        marker = dict(size=1.5, color=[f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in sample_colors], opacity=0.9)
    else:
        marker = dict(size=1.5, color=sample[:, 2], colorscale="Viridis", opacity=0.9)

    fig = go.Figure(data=[go.Scatter3d(x=sample[:, 0], y=sample[:, 1], z=sample[:, 2], mode="markers", marker=marker)])
    fig.update_layout(title=title, scene=dict(aspectmode="data"))
    fig.show()

def show_matplotlib(points: np.ndarray, colors: np.ndarray | None, title: str) -> None:
    sample = points
    sample_colors = colors
    max_points = 50_000
    if len(sample) > max_points:
        idx = np.random.choice(len(sample), max_points, replace=False)
        sample = sample[idx]
        sample_colors = sample_colors[idx] if sample_colors is not None else None

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    if sample_colors is not None and len(sample_colors) == len(sample):
        ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=sample_colors, s=0.2)
    else:
        ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=sample[:, 2], cmap="viridis", s=0.2)
    ax.set_title(title)
    ax.set_box_aspect((1, 1, 1))
    plt.show()

def show_point_cloud(path: Path, title: str) -> None:
    pcd = load_point_cloud(path)
    points, colors = pcd_to_arrays(pcd)
    print(title)
    print(pcd)
    try:
        o3d.visualization.draw([pcd], title=title, width=1024, height=768, show_ui=True, point_size=2.0)
    except Exception as exc:
        print(f"Open3D notebook viewer failed ({exc}); falling back to inline Plotly.")
        try:
            show_plotly(points, colors, title)
        except Exception as plotly_exc:
            print(f"Plotly visualization failed ({plotly_exc}); falling back to Matplotlib.")
            show_matplotlib(points, colors, title)

def describe_ply(path: Path) -> None:
    pcd = o3d.io.read_point_cloud(str(path))
    print(f"{path.name}: {len(pcd.points)} points")


In [ ]:
describe_ply(POINT_CLOUD_PLY)
show_point_cloud(POINT_CLOUD_PLY, "Gaussian map point cloud")


In [ ]:
describe_ply(GS_SPLAT_PLY)
show_point_cloud(GS_SPLAT_PLY, "Gaussian splat input cloud")


## Notes

- Open3D displays the xyz point data from both files; it does not render Gaussian-splat attributes in this notebook.
- If the Open3D notebook viewer is unavailable, the notebook falls back to inline Plotly, then Matplotlib.
- If the point cloud is not centered well in the first view, use the mouse wheel to zoom and the right mouse button to pan.
- If you want to inspect another run, change `RUN_NAME` in the discovery cell.
- The saved files live under `experiments_bash/results/test/<run-name>/gs_map/`.
